In [33]:
# Install required libraries
!pip install yt-dlp pydub

# Install ffmpeg (required)
!apt-get install -y ffmpeg


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


MAIN ALGORITHM FOR THE MASHUP

In [34]:
%%writefile mashup.py

import sys
import os
import shutil
import subprocess
from yt_dlp import YoutubeDL
from yt_dlp.utils import ExtractorError
from pydub import AudioSegment

# -------------------- Download Videos --------------------
def download_videos(singer, num_videos):
    print("\nDownloading videos...")

    search_query = f"ytsearch{num_videos}:{singer} songs"

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'videos/video_%(id)s.%(ext)s',
        'quiet': False,
        'noplaylist': True
    }

    with YoutubeDL(ydl_opts) as ydl:
        try:
            ydl.download([search_query])
        except ExtractorError as e:
            print(f"YouTubeDL Extractor Error: {e}")
            sys.exit(1)
        except Exception as e:
            print(f"Unexpected error during download: {e}")
            sys.exit(1)

    print("Download completed!")


# -------------------- Convert Videos to Audio --------------------
def convert_to_audio():
    print("\nConverting videos to audio...")
    os.makedirs("audios", exist_ok=True)

    for file in os.listdir("videos"):
        video_path = os.path.join("videos", file)
        audio_path = os.path.join("audios", file.split(".")[0] + ".mp3")

        command = [
            'ffmpeg',
            '-i', video_path,
            '-vn',
            '-acodec', 'libmp3lame',
            '-q:a', '2',
            audio_path
        ]

        subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print("Conversion completed!")


# -------------------- Cut Audio --------------------
def cut_audio(duration):
    print(f"\nCutting first {duration} seconds from each audio...")
    os.makedirs("cuts", exist_ok=True)

    for file in os.listdir("audios"):
        audio_path = os.path.join("audios", file)
        audio = AudioSegment.from_mp3(audio_path)

        cut_part = audio[:duration * 1000]
        cut_part.export(os.path.join("cuts", file), format="mp3")

    print("Audio cutting completed!")


# -------------------- Merge Audios --------------------
def merge_audios(output_file):
    print("\nMerging audio clips...")

    final_audio = AudioSegment.empty()

    for file in sorted(os.listdir("cuts")):
        audio_path = os.path.join("cuts", file)
        audio = AudioSegment.from_mp3(audio_path)
        final_audio += audio

    final_audio.export(output_file, format="mp3")
    print(f"\nFinal merged file saved as: {output_file}")


# -------------------- Clean Temporary Folders --------------------
def clean_folders():
    for folder in ["videos", "audios", "cuts"]:
        if os.path.exists(folder):
            shutil.rmtree(folder)


# -------------------- Main Function --------------------
def main():
    if len(sys.argv) != 5:
        print("Usage: python mashup.py \"Singer Name\" <NumberOfVideos> <Duration> <OutputFileName>")
        sys.exit(1)

    singer = sys.argv[1]
    num_videos = int(sys.argv[2])
    duration = int(sys.argv[3])
    output_file = sys.argv[4]

    if num_videos <= 10:
        print("Error: Number of videos must be greater than 10.")
        sys.exit(1)

    if duration <= 20:
        print("Error: Duration must be greater than 20 seconds.")
        sys.exit(1)

    if not output_file.endswith(".mp3"):
        print("Error: Output file must be .mp3")
        sys.exit(1)

    os.makedirs("videos", exist_ok=True)

    download_videos(singer, num_videos)
    convert_to_audio()
    cut_audio(duration)
    merge_audios(output_file)

    clean_folders()
    print("\nMashup Successfully Created!")


if __name__ == "__main__":
    main()


Overwriting mashup.py


In [35]:
!python mashup.py "Diljit Dosanjh" 11 25 diljit_mix.mp3



[youtube:search] Extracting URL: ytsearch11:Diljit Dosanjh songs
[download] Downloading playlist: Diljit Dosanjh songs
[youtube:search] query "Diljit Dosanjh songs": Downloading web client config
[youtube:search] query "Diljit Dosanjh songs" page 1: Downloading API JSON
[youtube:search] Playlist Diljit Dosanjh songs: Downloading 11 items of 11
[download] Downloading item 1 of 11
[youtube] Extracting URL: https://www.youtube.com/watch?v=WVq1siHnPxI
[youtube] WVq1siHnPxI: Downloading webpage
[youtube] WVq1siHnPxI: Downloading android vr player API JSON
[info] WVq1siHnPxI: Downloading 1 format(s): 251
[download] Destination: videos/video_WVq1siHnPxI.webm
[download] 100% of    2.73MiB in 00:00:00 at 17.60MiB/s
[download] Downloading item 2 of 11
[youtube] Extracting URL: https://www.youtube.com/watch?v=dCmp56tSSmA
[youtube] dCmp56tSSmA: Downloading webpage
[youtube] dCmp56tSSmA: Downloading android vr player API JSON
[info] dCmp56tSSmA: Downloading 1 format(s): 251
[download] Destination:

In [29]:
from google.colab import files
files.download("diljit_mix.mp3")


moviepy.editor import failed in a fresh cell.
This suggests an issue with the installation or environment.


to run for different singers we write
example
!python mashup.py "AP Dhillon" 8 25 ap_mix.mp3
files.download("ap_mix.mp3")
